# Tideway Models API Examples

Helper methods paired with direct calls for model CRUD, topology, and node queries. Reference: https://docs.bmc.com/xwiki/bin/view/IT-Operations-Management/Discovery/BMC-Helix-Discovery/DAAS/Integrating/Using-the-REST-APIs/Endpoints-in-the-REST-API/.

## Setup
- Install tideway (e.g. `pip install -e .` from repo root).
- Set `TARGET` and `TOKEN` below. Do **not** commit secrets.
- Adjust request bodies/ids per your appliance models.

In [ ]:

import tideway

# Configuration
TARGET = 'appliance-hostname-or-ip'  # e.g. 'discovery.example.com'
TOKEN = 'your-api-token'            # keep secrets out of source control
API_VERSION = '1.16'                # latest supported API
SSL_VERIFY = False                  # set to True when using valid certs

tw = tideway.appliance(TARGET, TOKEN, api_version=API_VERSION, ssl_verify=SSL_VERIFY)
models = tw.models()

def show_response(resp, label):
    if resp.ok:
        try:
            return resp.json()
        except Exception:
            return resp.text
    try:
        body = resp.json()
    except Exception:
        body = resp.text
    return {'endpoint': label, 'status': resp.status_code, 'body': body}

tw.api_about.json()  # quick sanity check


## List or filter models
Helper vs direct GET with optional filters.

In [ ]:

model_filters = {
    # 'name': 'Example',
    # 'type': 'business_service',
    # 'kind': 'custom',
}

models_helper = models.get_model(**model_filters)
models_direct = models.get('/models')

model_list_calls = {
    '/models (helper)': show_response(models_helper, '/models'),
    '/models (direct GET)': show_response(models_direct, '/models'),
}
model_list_calls


## Create, update, delete models
Provide bodies/ids to exercise helper vs direct calls.

In [ ]:

model_body = {
    # 'name': 'Example Model',
    # 'type': 'business_service',
    # 'definition': {},
}
model_key = ''  # set to a model key for patch/delete
model_patch_body = {
    # 'favorite': True,
}

model_calls = {}

if model_body:
    create_helper = models.post_model(model_body)
    create_direct = models.post('/models', model_body)
    model_calls['/models (helper POST)'] = show_response(create_helper, '/models')
    model_calls['/models (direct POST)'] = show_response(create_direct, '/models')

if model_key and model_patch_body:
    patch_helper = models.patch_model(model_key, model_patch_body)
    patch_direct = models.patch(f"/models/{model_key}", model_patch_body)
    model_calls[f"/models/{model_key} (helper PATCH)"] = show_response(patch_helper, f"/models/{model_key}")
    model_calls[f"/models/{model_key} (direct PATCH)"] = show_response(patch_direct, f"/models/{model_key}")

if model_key:
    delete_helper = models.delete_model(model_key)
    delete_direct = models.delete(f"/models/{model_key}")
    model_calls[f"/models/{model_key} (helper DELETE)"] = show_response(delete_helper, f"/models/{model_key}")
    model_calls[f"/models/{model_key} (direct DELETE)"] = show_response(delete_direct, f"/models/{model_key}")

model_calls or 'Set model_body/model_key to run create/update/delete.'


## Bulk/multi model operations
Use `/models/multi` for multi-action payloads.

In [ ]:

multi_body = {
    # 'create': [ ... ],
    # 'update': [ ... ],
}

if multi_body:
    multi_helper = models.post_model_multi(multi_body)
    multi_direct = models.post('/models/multi', multi_body)
    multi_calls = {
        '/models/multi (helper POST)': show_response(multi_helper, '/models/multi'),
        '/models/multi (direct POST)': show_response(multi_direct, '/models/multi'),
    }
else:
    multi_calls = 'Set multi_body to post /models/multi.'
multi_calls


## Model details by key
Helper vs direct for model definition and topology/node info.

In [ ]:

model_key_lookup = ''  # e.g. 'Model-1234'
attributes = None  # e.g. 'name,kind' for topology
node_kind = None    # e.g. 'Host'

if model_key_lookup:
    model_def_helper = models.get_model_key(model_key_lookup)
    model_def_direct = models.get(f"/models/{model_key_lookup}")

    if attributes:
        models.params['attributes'] = attributes
    topo_helper = models.get_model_topology(model_key_lookup, attributes=attributes)
    topo_direct = models.get(f"/models/{model_key_lookup}/topology")

    nodecount_helper = models.get_model_nodecount(model_key_lookup)
    nodecount_direct = models.get(f"/models/{model_key_lookup}/nodecount")

    nodes_helper = models.get_model_nodes(model_key_lookup, format='object', limit=25, kind=node_kind)
    nodes_direct = models.get(f"/models/{model_key_lookup}/nodes" + (f"/{node_kind}" if node_kind else ''))

    model_detail_calls = {
        f"/models/{model_key_lookup} (helper)": show_response(model_def_helper, f"/models/{model_key_lookup}"),
        f"/models/{model_key_lookup} (direct)": show_response(model_def_direct, f"/models/{model_key_lookup}"),
        f"/models/{model_key_lookup}/topology (helper)": show_response(topo_helper, f"/models/{model_key_lookup}/topology"),
        f"/models/{model_key_lookup}/topology (direct)": show_response(topo_direct, f"/models/{model_key_lookup}/topology"),
        f"/models/{model_key_lookup}/nodecount (helper)": show_response(nodecount_helper, f"/models/{model_key_lookup}/nodecount"),
        f"/models/{model_key_lookup}/nodecount (direct)": show_response(nodecount_direct, f"/models/{model_key_lookup}/nodecount"),
        f"/models/{model_key_lookup}/nodes (helper)": show_response(nodes_helper, f"/models/{model_key_lookup}/nodes"),
        f"/models/{model_key_lookup}/nodes (direct)": show_response(nodes_direct, f"/models/{model_key_lookup}/nodes"),
    }
else:
    model_detail_calls = 'Set model_key_lookup to inspect a model definition/topology/nodes.'
model_detail_calls


## Model lookup by node id
Helper vs direct for models tied to a node id.

In [ ]:

node_id = ''  # e.g. '1234567890abcdef'
expand_related = None  # e.g. True
node_kind = None       # optional kind filter for nodes

if node_id:
    model_by_node_helper = models.get_model_by_node_id(node_id, expand_related=expand_related)
    model_by_node_direct = models.get(f"/models/by_node_id/{node_id}")

    if expand_related:
        models.params['expand_related'] = expand_related
    topology_helper = models.get_topology_by_node_id(node_id)
    topology_direct = models.get(f"/models/by_node_id/{node_id}/topology")

    nodecount_helper = models.get_nodecount_by_node_id(node_id)
    nodecount_direct = models.get(f"/models/by_node_id/{node_id}/nodecount")

    nodes_helper = models.get_nodes_by_node_id(node_id, format='object', limit=25, kind=node_kind)
    nodes_direct = models.get(f"/models/by_node_id/{node_id}/nodes" + (f"/{node_kind}" if node_kind else ''))

    model_by_node_calls = {
        f"/models/by_node_id/{node_id} (helper)": show_response(model_by_node_helper, f"/models/by_node_id/{node_id}"),
        f"/models/by_node_id/{node_id} (direct)": show_response(model_by_node_direct, f"/models/by_node_id/{node_id}"),
        f"/models/by_node_id/{node_id}/topology (helper)": show_response(topology_helper, f"/models/by_node_id/{node_id}/topology"),
        f"/models/by_node_id/{node_id}/topology (direct)": show_response(topology_direct, f"/models/by_node_id/{node_id}/topology"),
        f"/models/by_node_id/{node_id}/nodecount (helper)": show_response(nodecount_helper, f"/models/by_node_id/{node_id}/nodecount"),
        f"/models/by_node_id/{node_id}/nodecount (direct)": show_response(nodecount_direct, f"/models/by_node_id/{node_id}/nodecount"),
        f"/models/by_node_id/{node_id}/nodes (helper)": show_response(nodes_helper, f"/models/by_node_id/{node_id}/nodes"),
        f"/models/by_node_id/{node_id}/nodes (direct)": show_response(nodes_direct, f"/models/by_node_id/{node_id}/nodes"),
    }
else:
    model_by_node_calls = 'Set node_id to inspect models tied to a node id.'
model_by_node_calls
